In [ ]:
# Análise Exploratória — Indicador Criança Alfabetizada

**Objetivo desta análise:** entender a estrutura, qualidade e distribuição da base
`features_alunos_ml` (camada Gold, Fase 2) antes de tomar qualquer decisão de
modelagem.

**Contexto da base:** cada linha é um aluno avaliado no 2º ano do Ensino
Fundamental, com proficiência no Saeb, contexto do seu município (taxa de
alfabetização agregada, meta vigente) e o rótulo `label_alfabetizado`
(1 = proficiência ≥ 743 pontos, 0 = caso contrário).

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_parquet("../data/raw/features_alunos_ml.parquet")
df.shape

(3867999, 15)

In [ ]:
## Estrutura e tipos de dados

Antes de qualquer análise de conteúdo, precisamos saber **como** cada coluna
está tipada — isso já revela problemas em potencial (ex.: uma coluna numérica
armazenada como texto, ou um identificador sendo tratado como número).

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3867999 entries, 0 to 3867998
Data columns (total 15 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   id_aluno                          object 
 1   ano                               int64  
 2   id_municipio                      object 
 3   nome                              object 
 4   sigla_uf                          object 
 5   nome_regiao                       object 
 6   rede                              object 
 7   rede_label                        object 
 8   serie                             object 
 9   proficiencia                      float64
 10  peso_aluno                        float64
 11  presenca                          object 
 12  taxa_alfabetizacao_municipio      float64
 13  meta_alfabetizacao_municipio_ano  float64
 14  label_alfabetizado                int32  
dtypes: float64(4), int32(1), int64(1), object(9)
memory usage: 427.9+ MB


In [ ]:
## Valores ausentes

Aqui medimos, por coluna, quantos valores estão ausentes em número absoluto e
em percentual do total. Isso vai definir, coluna por coluna, se a estratégia
certa é imputar, descartar a linha, ou descartar a coluna inteira (uma coluna
90% vazia não deveria ser imputada — deveria ser removida).

In [4]:
faltantes = df.isnull().sum().sort_values(ascending=False)
faltantes_pct = (faltantes / len(df) * 100).round(2)
pd.DataFrame({"faltantes": faltantes, "pct": faltantes_pct})

,faltantes,pct
meta_alfabetizacao_municipio_ano,1868546,48.31
proficiencia,513338,13.27
peso_aluno,513338,13.27
taxa_alfabetizacao_municipio,172460,4.46
id_aluno,0,0.00
ano,0,0.00
id_municipio,0,0.00
nome,0,0.00
sigla_uf,0,0.00
nome_regiao,0,0.00


In [6]:
df["label_alfabetizado"].value_counts()

label_alfabetizado
0    3867999
Name: count, dtype: int64

In [8]:
# Cruza com a regra oficial: quem tem proficiência >= 743 deveria ter label=1
pd.crosstab(df["proficiencia"] >= 743, df["label_alfabetizado"])

label_alfabetizado,0,1
proficiencia,,
False,1883453,0
True,0,1984546


In [ ]:
### Conclusão final e decisão de escopo

Com o rótulo corrigido, o padrão faz sentido: **100% dos alunos ausentes**
têm `label_alfabetizado = 0` — coerente com a própria definição da fonte
(não avaliado não pode ser considerado alfabetizado). Entre os alunos
**presentes**, a divisão é real: 59,14% alfabetizados vs. 40,86% não
alfabetizados — sem desbalanceamento severo, mas longe de ser 50/50.

**Decisão de escopo:** vamos restringir a modelagem aos alunos com
`presenca == 1`. Incluir os ausentes ensinaria o modelo a reaprender uma
regra determinística da própria fonte (ausência → não alfabetizado), não
um padrão preditivo real a partir de variáveis educacionais, territoriais
e socioeconômicas — que é exatamente o que o desafio pede. Essa é também a
primeira decisão concreta de **tratamento de data leakage** do projeto:
excluir uma variável (`presenca`) cujo valor 0 determina o rótulo por
construção, não por relação causal a ser aprendida.